<a href="https://colab.research.google.com/github/saagar-stha/FreeCodeCamp/blob/main/fcc_book_recommendation_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-09-07 10:35:48--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.2.33, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.5’

book-crossings.zip. 100%[===================>]  24.88M   151MB/s    in 0.2s    

2026-09-07 10:35:48 (151 MB/s) - ‘book-crossings.zip.5’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [4]:
# add your code here - consider creating a new cell for each section of code
# Filter to statistically significant users and books
user_counts = df_ratings['user'].value_counts()
valid_users = user_counts[user_counts >= 200].index

isbn_counts = df_ratings['isbn'].value_counts()
valid_isbns = isbn_counts[isbn_counts >= 100].index

df_ratings_filtered = df_ratings[
    df_ratings['user'].isin(valid_users) & df_ratings['isbn'].isin(valid_isbns)
]

# Merge with book titles, drop duplicate (isbn, title) rating rows if any
df = df_ratings_filtered.merge(df_books, on='isbn')
df = df.drop_duplicates(['title', 'user'])

# Pivot: rows = book title, columns = user, values = rating
book_user_matrix = df.pivot(index='title', columns='user', values='rating').fillna(0)

# Sparse matrix + model
book_matrix_sparse = csr_matrix(book_user_matrix.values)
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_matrix_sparse)

titles = book_user_matrix.index


def get_recommends(book=""):
    if book not in book_user_matrix.index:
        return [book, []]

    book_idx = book_user_matrix.index.get_loc(book)
    distances, indices = model.kneighbors(
        book_user_matrix.iloc[book_idx, :].values.reshape(1, -1),
        n_neighbors=6
    )

    recommended_books = []
    # skip index 0, that's the book itself (distance 0)
    for dist, idx in zip(distances[0][1:], indices[0][1:]):
        recommended_books.append([titles[idx], dist])

    # test expects farthest-of-the-5 first, nearest last
    recommended_books.reverse()

    return [book, recommended_books]

In [5]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", np.float32(0.8016211)], ['The Weight of Water', np.float32(0.77085835)], ['The Surgeon', np.float32(0.7699411)], ['I Know This Much Is True', np.float32(0.7677075)], ['The Lovely Bones: A Novel', np.float32(0.7234864)]]]
You passed the challenge! 🎉🎉🎉🎉🎉
